# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [1]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [5]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'llama3.2:1b'
ollama_url = "http://127.0.0.1:11434/v1"
ollama_key = "anything"
openai = OpenAI(base_url=ollama_url, api_key=ollama_key)

API key looks good so far


In [6]:
links = fetch_website_links("https://edwarddonner.com")
links

['https://edwarddonner.com/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2025/09/15/ai-in-production-gen-ai-and-agentic-ai-on-

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [2]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [3]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [4]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

https://edwarddonner.com/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.

In [7]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [11]:
rel_links = select_relevant_links("https://edwarddonner.com")

In [12]:
rel_links

{'links': [{'type': 'about page', 'url': 'https://edwarddonner.com/about'},
  {'type': 'careers page', 'url': 'https://edwarddonner.com/careers'},
  {'type': 'news',
   'url': 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html'}]}

In [13]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [14]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling llama3.2:1b
Found 2 relevant links


{'links': [{'type': 'about page', 'url': 'https://edwarddonner.com/about'},
  {'type': 'profile page', 'url': 'mailto:hello@mygroovydomain.com'}]}

In [15]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling llama3.2:1b
Found 2 relevant links


{'links': [{'type': 'about page', 'url': 'https://huggingface.co/about'},
  {'type': 'enterprise', 'url': 'https://huggingface.co/pricing#endpoints'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [16]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [18]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling llama3.2:1b
Found 3 relevant links
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Website
Tasks
HuggingChat
Collections
Languages
Organizations
Community
Blog
Posts
Daily Papers
Learn
Discord
Forum
GitHub
Solutions
Team & Enterprise
Hugging Face PRO
Enterprise Support
Inference Providers
Inference Endpoints
Storage Buckets
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
SulphurAI/Sulphur-2-base
Updated
3 days ago
•
1.11M
•
1.19k
openbmb/MiniCPM-V-4.6
Updated
about 17 hours ago
•
145k
•
810
bytedance-research/Lance
Updated
about 4 hours ago
•
171
•
353
Supertone/supertonic-3
Updated
2 days ago
•
28.7k
•
475
unsloth/Qwen3.6-27B-MTP-GGUF
Updated
2 days ago
•
337

In [19]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [20]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [22]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling llama3.2:1b
Found 2 relevant links


"\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nBuckets\nnew\nDocs\nEnterprise\nPricing\nWebsite\nTasks\nHuggingChat\nCollections\nLanguages\nOrganizations\nCommunity\nBlog\nPosts\nDaily Papers\nLearn\nDiscord\nForum\nGitHub\nSolutions\nTeam & Enterprise\nHugging Face PRO\nEnterprise Support\nInference Providers\nInference Endpoints\nStorage Buckets\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nSulphurAI/Sulphur-2-base\nUpdated\n3 days ago\n•\n1.11M\n•\n1.19k\nopenbmb/MiniCPM-V-4.6\nUpdated\nabout 17 hours ago\n•\n145k

In [24]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [26]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling llama3.2:1b
Found 3 relevant links


Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.


## Hugging Face - The AI Community Building the Future


### About Us

Hugging Face is a leading provider of artificial intelligence (AI) models, datasets, and applications. We are committed to making AI more accessible and collaborative for the machine learning community.

### Solutions

Our platform offers solutions to help businesses and individuals leverage the power of AI in their workflows. With our range of modules, you can easily integrate AI-powered solutions into your existing processes.


#### Cloud Architectures


• **T4**: Build applications that run fast and scalable on a cloud-based infrastructure.
• **GPU T4**: Accelerate your machine learning models with dedicated GPU acceleration.
• **MCP T4**: Optimize your TensorFlow or PyTorch model with zero latency and high performance.

### Collaborative Platform

Hugging Face's collaborative platform enables the community to share, discover, and reuse AI models. Explore our collection of 2M+ models and discover new opportunities for collaboration:


#### Models


Browse an extensive range of AI models, from image-to-image translation to natural language processing.


#### Datasets


 Discover datasets collected by experts in various domains, providing valuable insights into real-world scenarios.


#### Spaces

Collaborate with other developers on projects or discuss your ideas in our community forum.

#### Inference Endpoints

Integrate our inference endpoints into your applications for efficient and secure AI inference.
Browse 2M+ models
Trending on
this week
Models
SulphurAI/Sulphur-2-base
Updated
3 days ago
•
1.11M
•
1.19k
openbmb/MiniCPM-V-4.6

### Enterprise Solutions

Hugging Face offers enterprise-level solutions and support to help companies capitalize on AI opportunities.

#### Inference Providers

Integrate Hugging Face's inference providers into your applications for high-throughput data processing.
Inference Endpoints
Catalog
Log In
· 
· 
·
·
·
·
·
·
·
·

### Training and Collaboration

Collaborate with our community on projects, propose new models or datasets, or simply explore how you can leverage Hugging Face's resources.


#### Enterprise Support

Access expert-level support for optimizing performance, security, and compliance in your AI applications.
Enterprise Support
Inference Providers


| Module          | Performance    | Security      | Compliant      |
|-----------------|-----------------|---------------|
| Sulphur-2-base  | High            | High         | High          |

#### T4 Cloud Services

Utilize our cloud infrastructure to build scalable and efficient machine learning applications.
T4 Cloud Services
Cloud Architectures



## How We Work Together

* **Collaborate on projects**: Share ideas, resources, and expertise with the community to create innovative AI solutions.
* **Propose new models or datasets**: Help us improve our model offerings by sharing your work or proposing a dataset for inclusion.
* **Ask questions or get help**: Reach out to our experts via email or discuss your needs online.

### Join Our Community


Learn from others, share your knowledge, and be part of the Hugging Face community:

#### Learn

Explore our tutorials, guides, and documentation to improve your AI skills.
Discussions
Daily Papers


### Get Started Today

Sign up for a free account today and start exploring our resources:


· 
·
·
·
|
•
!

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [27]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [30]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling llama3.2:1b
Found 3 relevant links


# Hugging Face: The AI Community Building the Future

## About Us

Hugging Face is an open-source organization that provides a collaborative platform for machine learning (ML) research and development. Our mission is to democratize access to AI technologies, making them more accessible and user-friendly.

## Models

* **2M+ Model Collection**: Hugging Face hosts a vast library of pre-trained models in various NLP and computer vision tasks.
	+ **SulphurAI/Sulphur-2-base**: A transformer-based model for text classification.
	+ **openbmb/MiniCPM-V-4.6**: A multi-label classification classifier optimized for performance at-scale.
* **Datasets**: Hugging Face offers a wide range of datasets, covering various domains and use cases.
	+ **AlienKevin/SWE-ZERO-12M-trajectories**: Trajectory data for object detection applications.

## Spaces and Buckets

Hugging Face's collaborative platform allows users to host and share models in various spaces and buckets.

* **Running on Zero Agents**: Many of our models run on low-latency agents, making them suitable for real-time applications.
* **T4/MCP**: Our models are designed to utilize T4 and MCP models (Meta-PowerPoint Classification) for better performance in certain scenarios.

## Applications

Hugging Face's platform enables users to explore and apply AI technologies across various applications.

* **Image-to-3D Generation**: Pixal3D uses Hugging Face's Zero Model to generate high-fidelity 3D images from real-world objects.
* **Video Text Extraction**: Gemma-4-E4B-Uncensored-HauhauCS-Aggressive-Q5 uses T4 models for text extraction tasks.

## Learn and Explore

For those interested in learning more, we have:

* **Blog Posts**: Hugging Face's blog provides updates on our projects, research, and community news.
* **Daily Papers**: Our paper archive showcases research contributions from across the AI community.


RemoteProtocolError: peer closed connection without sending complete message body (incomplete chunked read)

In [ ]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>